# Retail Intelligence Platform

## Feature Engineering

### Objectives

- Create new business features
- Improve data quality
- Prepare dataset for Machine Learning
- Save processed dataset

In [1]:
import pandas as pd

pd.set_option("display.max_columns", None)

In [2]:
df = pd.read_csv("../data/cleaned/cleaned_superstore.csv")

df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Order Year,Order Month,Order Month Name,Order Quarter,Order Day,Order Weekday,Shipping Days
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136,2016,11,November,4,8,Tuesday,3
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820,2016,11,November,4,8,Tuesday,3
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714,2016,6,June,2,12,Sunday,4
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310,2015,10,October,4,11,Sunday,7
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164,2015,10,October,4,11,Sunday,7


In [3]:
df["Profit Margin (%)"] = (df["Profit"] / df["Sales"]) * 100

df["Profit Margin (%)"] = df["Profit Margin (%)"].round(2)

df[["Sales", "Profit", "Profit Margin (%)"]].head()

,Sales,Profit,Profit Margin (%)
0,261.9600,41.9136,16.00
1,731.9400,219.5820,30.00
2,14.6200,6.8714,47.00
3,957.5775,-383.0310,-40.00
4,22.3680,2.5164,11.25


In [4]:
df["Sales Per Unit"] = (df["Sales"] / df["Quantity"]).round(2)

df[["Sales", "Quantity", "Sales Per Unit"]].head()

,Sales,Quantity,Sales Per Unit
0,261.9600,2,130.98
1,731.9400,3,243.98
2,14.6200,2,7.31
3,957.5775,5,191.52
4,22.3680,2,11.18


In [5]:
def discount_category(discount):
    if discount == 0:
        return "No Discount"
    elif discount <= 0.2:
        return "Low"
    elif discount <= 0.5:
        return "Medium"
    else:
        return "High"

df["Discount Category"] = df["Discount"].apply(discount_category)

df[["Discount", "Discount Category"]].head()

,Discount,Discount Category
0,0.00,No Discount
1,0.00,No Discount
2,0.00,No Discount
3,0.45,Medium
4,0.20,Low


In [6]:
def order_size(quantity):
    if quantity <= 2:
        return "Small"
    elif quantity <= 5:
        return "Medium"
    else:
        return "Large"

df["Order Size"] = df["Quantity"].apply(order_size)

df[["Quantity", "Order Size"]].head()

,Quantity,Order Size
0,2,Small
1,3,Medium
2,2,Small
3,5,Medium
4,2,Small


In [7]:
def sales_category(sales):
    if sales < 100:
        return "Low Sales"
    elif sales < 500:
        return "Medium Sales"
    else:
        return "High Sales"

df["Sales Category"] = df["Sales"].apply(sales_category)

df[["Sales", "Sales Category"]].head()

,Sales,Sales Category
0,261.9600,Medium Sales
1,731.9400,High Sales
2,14.6200,Low Sales
3,957.5775,High Sales
4,22.3680,Low Sales


In [8]:
def profit_category(profit):
    if profit < 0:
        return "Loss"
    elif profit < 100:
        return "Low Profit"
    elif profit < 500:
        return "Medium Profit"
    else:
        return "High Profit"

df["Profit Category"] = df["Profit"].apply(profit_category)

df[["Profit", "Profit Category"]].head()

,Profit,Profit Category
0,41.9136,Low Profit
1,219.5820,Medium Profit
2,6.8714,Low Profit
3,-383.0310,Loss
4,2.5164,Low Profit


In [9]:
def shipping_speed(days):
    if days <= 2:
        return "Fast"
    elif days <= 5:
        return "Standard"
    else:
        return "Slow"

df["Shipping Speed"] = df["Shipping Days"].apply(shipping_speed)

df[["Shipping Days", "Shipping Speed"]].head()

,Shipping Days,Shipping Speed
0,3,Standard
1,3,Standard
2,4,Standard
3,7,Slow
4,7,Slow


In [10]:
aov = df.groupby("Order ID")["Sales"].sum().mean()

print("Average Order Value:", round(aov, 2))

Average Order Value: 458.61


In [11]:
clv = (
    df.groupby("Customer ID")["Sales"]
      .sum()
      .reset_index(name="Customer Lifetime Value")
)

clv.head()

,Customer ID,Customer Lifetime Value
0,AA-10315,5563.560
1,AA-10375,1056.390
2,AA-10480,1790.512
3,AA-10645,5086.935
4,AB-10015,886.156


In [12]:
df = df.merge(clv, on="Customer ID", how="left")

df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Order Year,Order Month,Order Month Name,Order Quarter,Order Day,Order Weekday,Shipping Days,Profit Margin (%),Sales Per Unit,Discount Category,Order Size,Sales Category,Profit Category,Shipping Speed,Customer Lifetime Value
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136,2016,11,November,4,8,Tuesday,3,16.00,130.98,No Discount,Small,Medium Sales,Low Profit,Standard,1148.7800
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820,2016,11,November,4,8,Tuesday,3,30.00,243.98,No Discount,Medium,High Sales,Medium Profit,Standard,1148.7800
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714,2016,6,June,2,12,Sunday,4,47.00,7.31,No Discount,Small,Low Sales,Low Profit,Standard,1119.4830
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310,2015,10,October,4,11,Sunday,7,-40.00,191.52,Medium,Medium,High Sales,Loss,Slow,2602.5755
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164,2015,10,October,4,11,Sunday,7,11.25,11.18,Low,Small,Low Sales,Low Profit,Slow,2602.5755


In [13]:
new_columns = [
    "Profit Margin (%)",
    "Sales Per Unit",
    "Discount Category",
    "Order Size",
    "Sales Category",
    "Profit Category",
    "Shipping Speed",
    "Customer Lifetime Value"
]

df[new_columns].head()

,Profit Margin (%),Sales Per Unit,Discount Category,Order Size,Sales Category,Profit Category,Shipping Speed,Customer Lifetime Value
0,16.00,130.98,No Discount,Small,Medium Sales,Low Profit,Standard,1148.7800
1,30.00,243.98,No Discount,Medium,High Sales,Medium Profit,Standard,1148.7800
2,47.00,7.31,No Discount,Small,Low Sales,Low Profit,Standard,1119.4830
3,-40.00,191.52,Medium,Medium,High Sales,Loss,Slow,2602.5755
4,11.25,11.18,Low,Small,Low Sales,Low Profit,Slow,2602.5755


In [14]:
df.to_csv("../data/processed/processed_superstore.csv", index=False)

print("Processed dataset saved successfully.")

Processed dataset saved successfully.


In [15]:
print("="*50)
print("FEATURE ENGINEERING COMPLETED")
print("="*50)

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nNew Features Added:")

for col in [
    "Profit Margin (%)",
    "Sales Per Unit",
    "Discount Category",
    "Order Size",
    "Sales Category",
    "Profit Category",
    "Shipping Speed",
    "Customer Lifetime Value"
]:
    print("✔", col)

FEATURE ENGINEERING COMPLETED
Rows: 9994
Columns: 36

New Features Added:
✔ Profit Margin (%)
✔ Sales Per Unit
✔ Discount Category
✔ Order Size
✔ Sales Category
✔ Profit Category
✔ Shipping Speed
✔ Customer Lifetime Value
